In [8]:
import pandas as pd
import ast
import snappy

In [9]:
df = pd.read_csv('knotinfo.csv')

def fix_pd_notation(pd_notation):
    return pd_notation.replace(';', ',')

NUM_AUGMENTATIONS = 10

augmented_rows = []
skipped = 0

for idx in range(len(df)):
    pd_notation = df['PD Notation'].iloc[idx]
    if pd.isna(pd_notation):
        skipped += 1
        continue

    pd_str = fix_pd_notation(str(pd_notation))
    pd_list = ast.literal_eval(pd_str)

    for _ in range(NUM_AUGMENTATIONS):
        try:
            link = snappy.Link(pd_list)
            link.backtrack(steps=10, prob_type_1=0.3, prob_type_2=0.3)

            new_row = df.iloc[idx].copy()
            new_row['PD Notation'] = str(link.PD_code())
            augmented_rows.append(new_row)
        except Exception as e:
            if skipped < 3:
                print(f"Row {idx} failed: {e}")
            skipped += 1

print(f"Augmented {len(augmented_rows)} knots, skipped {skipped}")

aug_df = pd.concat([df, pd.DataFrame(augmented_rows)], ignore_index=True)
print(f"Original: {len(df)}, Augmented total: {len(aug_df)}")
aug_df.to_csv('knotinfo_aug.csv', index=False)

Augmented 69000 knots, skipped 0
Original: 6900, Augmented total: 75900
